# 8-MEMS Gyroscope Multi-Algorithm Sensor Fusion

This notebook evaluates multiple estimators on one canonical synchronized sensor realization. The baseline stream is an evaluation reference, not a truth source. RTS results are explicitly offline-only.

## 2. System Architecture
Gyro 1-8 use ports 5001-5008 and the baseline uses 5009 on 127.0.0.1.

Streaming or recorded input -> timestamp synchronization -> one canonical ExperimentDataset -> multiple algorithms -> common evaluation.

Each algorithm receives the same `dataset.gyro_data`; the baseline is not an input sensor.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import NUM_GYROS, SAMPLE_RATE, RAW_CSV, BASELINE_RAW_CSV, RANDOM_SEED
from algorithms.acquisition import collect_streams_once, align_streams
from algorithms.dataset import ExperimentDataset, validate_dataset, apply_degradation
from algorithms.inverse_covariance import inverse_covariance_fusion
from algorithms.central_kf import run_central_kf
from algorithms.federated_kf import run_federated_kf
from algorithms.covariance_intersection import covariance_intersection
from algorithms.adaptive_kf import run_adaptive_kf
from algorithms.rts_smoother import rts_smooth
from algorithms.evaluation import correlation_matrices, comparison_table, error_metrics, latency_summary, nis

RUN_LIVE = False
LIVE_SAMPLE_COUNT = 1000
MAX_PROCESSING_SAMPLES = 2000
LIVE_TOLERANCE = 1.0 / SAMPLE_RATE / 2.0
EXPERIMENT_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'results' / 'experiments' / EXPERIMENT_ID
for folder in ('raw', 'processed', 'metrics', 'plots', 'tables', 'logs'):
    (OUTPUT_ROOT / folder).mkdir(parents=True, exist_ok=True)
print({'experiment_id': EXPERIMENT_ID, 'sample_rate_hz': SAMPLE_RATE, 'random_seed': RANDOM_SEED, 'run_live': RUN_LIVE, 'max_processing_samples': MAX_PROCESSING_SAMPLES})

## 3. Configuration
The simulator configuration is imported from `config.py`. `LIVE_SAMPLE_COUNT` is an experiment-layer setting because the existing configuration has no live batch-size constant.

## 4. Live Data Acquisition
When `RUN_LIVE` is enabled, all nine existing streams are read once concurrently through `GyroStreamClient`. Embedded sensor timestamps remain authoritative; network arrival time is not used.

In [ ]:
if RUN_LIVE:
    streams = collect_streams_once(range(5001, 5010), LIVE_SAMPLE_COUNT, host='127.0.0.1')
    aligned_frame = align_streams(streams, tolerance=LIVE_TOLERANCE)
    gyro_columns = ['timestamp'] + [f'gyro{i}_{axis}' for i in range(1, NUM_GYROS + 1) for axis in 'xyz']
    baseline_columns = ['timestamp', 'baseline_x', 'baseline_y', 'baseline_z']
    dataset = ExperimentDataset.from_frames(aligned_frame[gyro_columns], aligned_frame[baseline_columns], sample_rate=SAMPLE_RATE, experiment_id=EXPERIMENT_ID)
else:
    dataset = ExperimentDataset.from_csv(PROJECT_ROOT / RAW_CSV, PROJECT_ROOT / BASELINE_RAW_CSV, sample_rate=SAMPLE_RATE, experiment_id=EXPERIMENT_ID)

if dataset.sample_count > MAX_PROCESSING_SAMPLES:
    stride = int(np.ceil(dataset.sample_count / MAX_PROCESSING_SAMPLES))
    dataset = ExperimentDataset(
        dataset.timestamps[::stride], dataset.gyro_data[::stride],
        None if dataset.baseline_data is None else dataset.baseline_data[::stride],
        None if dataset.truth_data is None else dataset.truth_data[::stride],
        dataset.sample_rate / stride if dataset.sample_rate else None, dataset.experiment_id,
    )

dataset.to_frame().to_csv(OUTPUT_ROOT / 'raw' / 'canonical_synchronized_dataset.csv', index=False)
print('One canonical dataset created:', dataset.sample_count, 'samples;', dataset.sensor_count, 'sensors')

## 5. Data Validation
Validation checks stream count, timestamp ordering, duplicate timestamps, finite values, sample interval, duration, and optional baseline/truth presence.

In [ ]:
validation = validate_dataset(dataset)
validation['status'] = 'PASS' if validation['finite'] and not validation['duplicate_timestamps'] and validation['sensor_count'] == NUM_GYROS else 'FAIL'
print(pd.Series(validation).to_string())
assert validation['status'] == 'PASS'

## 6. Canonical Synchronized Dataset
All subsequent cells use this exact object. No algorithm regenerates sensor data.

In [ ]:
print(f'samples={dataset.sample_count}, duration_s={dataset.duration:.6f}, mean_dt_s={dataset.mean_dt:.9f}')
print(f'min_dt_s={validation["min_dt"]:.9f}, max_dt_s={validation["max_dt"]:.9f}')
print('alignment errors are measured against embedded timestamps; arrival time is intentionally excluded')

## 7. Sensor Correlation
Raw correlation includes common true motion. Residual correlation is only computed when truth is available; the baseline is never substituted as truth for this analysis.

In [ ]:
raw_correlations = correlation_matrices(dataset.gyro_data)
residual_correlations = correlation_matrices(dataset.gyro_data, dataset.truth_data) if dataset.truth_data is not None else None
for name, matrices in [('raw', raw_correlations), ('residual', residual_correlations)]:
    if matrices is None:
        print(name, 'correlation: NOT AVAILABLE without truth')
        continue
    for axis, matrix in matrices.items():
        pd.DataFrame(matrix).to_csv(OUTPUT_ROOT / 'metrics' / f'{name}_correlation_{axis}.csv', index=False)
        pairwise = matrix[np.triu_indices_from(matrix, k=1)]
        print(name, axis, 'mean/min/max=', np.mean(pairwise), np.min(pairwise), np.max(pairwise))

## 8. Common Signal Visualization
Representative raw sensors are shown using actual embedded timestamps.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for axis_index, axis in enumerate('xyz'):
    axes[axis_index].plot(dataset.timestamps, dataset.gyro_data[:, 0, axis_index], label='gyro1')
    axes[axis_index].plot(dataset.timestamps, dataset.gyro_data[:, -1, axis_index], alpha=0.7, label='gyro8')
    if dataset.truth_data is not None:
        axes[axis_index].plot(dataset.timestamps, dataset.truth_data[:, axis_index], label='truth')
    axes[axis_index].set_ylabel(axis); axes[axis_index].legend()
axes[-1].set_xlabel('sensor timestamp (s)'); fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'plots' / 'raw_gyro_signals.png', dpi=150); plt.show()

## 9. Mean Fusion
Arithmetic mean is the existing simple fusion baseline.

In [ ]:
mean_estimate = dataset.gyro_data.mean(axis=1)
print('mean fusion shape:', mean_estimate.shape)

## 10. Inverse-Covariance Fusion
Weights use per-sensor/per-axis variance estimates. This is a benchmark that assumes independent errors or a known covariance structure and can be overconfident for this correlated array.

In [ ]:
variance = np.maximum(np.var(np.diff(dataset.gyro_data, axis=0), axis=0) / 2.0, 1e-10)
inverse_estimate, inverse_covariance, inverse_weights = inverse_covariance_fusion(dataset.gyro_data, variance)
print('mean weights by axis:', inverse_weights.mean(axis=0))

## 11. Central KF
The current random-walk measurement model is linear, so this is documented as KF rather than EKF. The implementation uses a Joseph covariance update and independent diagonal sensor measurement covariance.

In [ ]:
central = run_central_kf(dataset.gyro_data, dataset.timestamps, variance, process_covariance=1e-5)
central_estimate = central['estimate']
print('central KF shape:', central_estimate.shape)

## 12. Federated Kalman Filter
Each gyro has an independent local filter, followed by information-form master fusion. Because common errors can be correlated, the independent-information result is explicitly an approximation and may double-count common information.

In [ ]:
federated = run_federated_kf(dataset.gyro_data, dataset.timestamps, variance, process_variance=1e-5)
federated_estimate = federated['estimate']
print('federated KF shape:', federated_estimate.shape)

## 13. Covariance Intersection
Sequential CI fuses sensor estimates without assuming known cross-correlation. The weight minimizes trace covariance on a grid and is not an exact posterior for correlated sensors.

In [ ]:
ci_estimates, ci_covariances, ci_weights = [], [], []
sensor_covariances = np.array([np.diag(variance[sensor]) for sensor in range(NUM_GYROS)])
ci_stride = max(1, int(np.ceil(dataset.sample_count / 2000)))
ci_input = dataset.gyro_data[::ci_stride]
for sample in ci_input:
    current_x, current_p = sample[0], sensor_covariances[0]
    sample_weights = []
    for sensor in range(1, NUM_GYROS):
        current_x, current_p, weight = covariance_intersection(current_x, current_p, sample[sensor], sensor_covariances[sensor])
        sample_weights.append(weight)
    ci_estimates.append(current_x); ci_covariances.append(current_p); ci_weights.append(sample_weights)
ci_estimate = np.asarray(ci_estimates); ci_covariance = np.asarray(ci_covariances)
print('CI shape:', ci_estimate.shape, 'mean weights:', np.mean(ci_weights, axis=0))

## 14. Adaptive Kalman Filter
The effective measurement variance adapts from innovation energy with a documented forgetting factor and adaptation gain. It is not a random Q/R mutation.

In [ ]:
adaptive = run_adaptive_kf(mean_estimate, dataset.timestamps)
adaptive_estimate = adaptive['estimate']
print('adaptive final measurement variance:', adaptive['measurement_variance'][-1])

## 15. RTS Smoother
RTS uses the stored forward KF results and a backward pass. It is OFFLINE ONLY and is excluded from causal real-time latency ranking.

In [ ]:
rts = rts_smooth(central['estimate'], central['covariance'], dataset.timestamps)
rts_estimate = rts['estimate']
print('RTS OFFLINE ONLY shape:', rts_estimate.shape)

## 16. Accuracy Evaluation
Truth-based metrics are used only when simulator truth is available. Otherwise the baseline is reported as a practical reference and is not described as ground truth.

In [ ]:
reference = dataset.truth_data if dataset.truth_data is not None else dataset.baseline_data
reference_name = 'truth' if dataset.truth_data is not None else 'baseline_reference'
estimates = {'mean': mean_estimate, 'inverse_covariance': inverse_estimate, 'KF': central_estimate, 'federated_KF': federated_estimate, 'CI': ci_estimate, 'adaptive_KF': adaptive_estimate, 'RTS_OFFLINE_ONLY': rts_estimate}
comparison = comparison_table(estimates, reference, SAMPLE_RATE)
comparison['reference'] = reference_name
comparison.to_csv(OUTPUT_ROOT / 'tables' / 'algorithm_comparison.csv', index=False)
display(comparison)

## 17. Drift / Bias Evaluation
The comparison table includes mean bias, standard deviation, maximum absolute error, and drift converted to deg/hour only as a numeric time-rate diagnostic. The 1.7-2.3 deg/hour target is experimental and is not asserted comparable without a unit/calibration review.

## 18. Latency Evaluation
Processing latency uses `time.perf_counter()` around each algorithm call. Acquisition and network timing are separate concerns; RTS is not ranked as causal.

In [ ]:
calls = {
    'mean': lambda: dataset.gyro_data.mean(axis=1),
    'inverse_covariance': lambda: inverse_covariance_fusion(dataset.gyro_data, variance),
    'KF': lambda: run_central_kf(dataset.gyro_data, dataset.timestamps, variance),
    'federated_KF': lambda: run_federated_kf(dataset.gyro_data, dataset.timestamps, variance),
    'adaptive_KF': lambda: run_adaptive_kf(mean_estimate, dataset.timestamps),
}
latency = pd.DataFrame([{'algorithm': name, **latency_summary(call, repeats=3)} for name, call in calls.items()])
latency.to_csv(OUTPUT_ROOT / 'metrics' / 'processing_latency.csv', index=False)
display(latency)

## 19. Consistency
NIS is computed only for the central KF, which provides innovations and innovation covariance. NEES requires a known truth state and is not fabricated when truth is absent.

In [ ]:
central_nis = nis(central['innovation'], central['innovation_covariance'])
pd.DataFrame({'nis': central_nis}).to_csv(OUTPUT_ROOT / 'metrics' / 'central_kf_nis.csv', index=False)
print('central KF NIS mean:', float(np.mean(central_nis)))
print('NEES:', 'available only when a validated truth state is supplied')

## 20. Monte Carlo
Every trial uses one controlled realization shared by every algorithm. This compact runner is configurable and avoids changing the original simulator.

In [ ]:
def monte_carlo_mean_vs_inverse(trials=3, samples=200, sensors=NUM_GYROS, seed=RANDOM_SEED):
    rows = []
    for trial in range(trials):
        rng = np.random.default_rng(seed + trial)
        truth = rng.normal(0, 1, (samples, 3))
        realization = truth[:, None, :] + rng.normal(0, 0.02, (samples, sensors, 3))
        variance_mc = np.full((sensors, 3), 0.02 ** 2)
        mean_mc = realization.mean(axis=1)
        inverse_mc, _, _ = inverse_covariance_fusion(realization, variance_mc)
        for name, estimate in [('mean', mean_mc), ('inverse_covariance', inverse_mc)]:
            rows.append({'trial': trial, 'algorithm': name, **error_metrics(estimate, truth)})
    return pd.DataFrame(rows)

monte_carlo = monte_carlo_mean_vs_inverse()
monte_carlo.to_csv(OUTPUT_ROOT / 'tables' / 'monte_carlo.csv', index=False)
display(monte_carlo)

## 21. Algorithm Comparison
The table is generated from the common reference and common dataset. It does not declare a universal winner.

## 22. Accuracy vs Latency
Only causal algorithms with measured processing latency are plotted.

In [ ]:
plot_table = comparison.merge(latency[['algorithm', 'mean_ms', 'p95_ms', 'p99_ms']], on='algorithm', how='left')
plot_table.to_csv(OUTPUT_ROOT / 'tables' / 'comparison_with_latency.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(plot_table['mean_ms'], plot_table['rmse'])
for _, row in plot_table.iterrows():
    if np.isfinite(row['mean_ms']): ax.annotate(row['algorithm'], (row['mean_ms'], row['rmse']))
ax.set_xlabel('processing latency mean (ms)'); ax.set_ylabel('RMSE'); ax.set_title('Accuracy vs latency')
fig.tight_layout(); fig.savefig(OUTPUT_ROOT / 'plots' / 'accuracy_vs_latency.png', dpi=150); plt.show()

## 23. Robustness Experiments
Scenarios are configured in the experiment layer and do not modify the simulator. Supported examples include noise, bias, corrupted intervals, and missing-sample replacement policy.

In [ ]:
SCENARIOS = {'nominal': None, 'gyro3_bias': {'sensor': 2, 'bias': [0.05, 0.0, 0.0]}, 'gyro5_noise': {'sensor': 4, 'noise_scale': 0.05}}
robustness = {name: validate_dataset(apply_degradation(dataset, scenario, RANDOM_SEED)) for name, scenario in SCENARIOS.items()}
display(pd.DataFrame(robustness).T)

## 24. Final Summary
Outputs are written below the unique experiment directory and do not overwrite prior research runs.

In [ ]:
summary = {
    'experiment_id': EXPERIMENT_ID, 'sample_count': dataset.sample_count, 'duration_s': dataset.duration,
    'sample_rate_hz': SAMPLE_RATE, 'reference': reference_name, 'algorithms': list(estimates),
    'rts_status': 'OFFLINE_ONLY', 'residual_correlation': 'available' if residual_correlations is not None else 'not available without truth',
    'scientific_status': 'IMPLEMENTED AND SMOKE-TESTED; RESULTS ARE NOT CONCLUSIVE',
}
(OUTPUT_ROOT / 'logs' / 'final_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

## 25. Conclusions
Interpret results only for this synchronized realization and configuration. Correlated sensor errors limit independent inverse-covariance and federated assumptions; CI is conservative when cross-correlation is unknown; RTS is not causal; and baseline comparisons are not truth validation unless simulator truth is explicitly present.